# Bloc 5 — Stochastic Processes & Numerical Approximation
## Approximation numérique et simulation efficace

## Étape 1 — Deterministic vs Monte Carlo Approximation
### Approximation déterministe et stochastique

### Partie A — Mise en pratique

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

X = joblib.load(
    "data_processed/X_block3_weather_nonlinear.joblib"
)

y = joblib.load(
    "data_processed/y_block3_weather.joblib"
)

model = joblib.load(
    "models/block3_conformal_ridge.joblib"
)

# 2024 = calibration
cal = (
    (X.index >= "2024-01-01")
    & (X.index < "2025-01-01")
)

# 2025 = validation
val = (
    (X.index >= "2025-01-01")
    & (X.index < "2026-01-01")
)

cal_index = X.loc[cal].index
val_index = X.loc[val].index

pred_cal = model.predict(X.loc[cal])
pred_val = model.predict(X.loc[val])

y_cal = y.loc[cal].values
y_val = y.loc[val].values

residual_cal = y_cal - pred_cal

# Résidus 2024 sous forme de journées de 48 demi-heures
resid_df = pd.DataFrame(
    {"residual": residual_cal},
    index=cal_index
)

resid_df["date"] = resid_df.index.date
resid_df["slot"] = (
    resid_df.index.hour * 2
    + resid_df.index.minute // 30
)

daily_residuals = (
    resid_df
    .pivot(
        index="date",
        columns="slot",
        values="residual"
    )
    .dropna()
)

R = daily_residuals.to_numpy()

# Journée 2025 avec le plus grand pic prévu
pred_series = pd.Series(
    pred_val,
    index=val_index
)

selected_day = (
    pred_series.resample("D")
    .max()
    .idxmax()
)

day_mask = (
    val_index.normalize()
    == selected_day.normalize()
)

pred_day = pred_val[day_mask]
y_day = y_val[day_mask]

print("Jours calibration :", R.shape[0])
print("Variables/jour :", R.shape[1])
print("Journée étudiée :", selected_day)
print("Pic réel :", y_day.max())

Jours calibration : 366
Variables/jour : 48
Journée étudiée : 2025-01-14 00:00:00+00:00
Pic réel : 87716.0


In [2]:
mean_residual_slot = R.mean(axis=0)

deterministic_peak = np.max(
    pred_day + mean_residual_slot
)

print(
    "Deterministic peak :",
    deterministic_peak
)

Deterministic peak : 87882.06658735783


In [3]:
rng = np.random.default_rng(42)

N_SIM = 5000

U = rng.random(
    (N_SIM, 48)
)

residuals_mc = np.empty_like(U)

for j in range(48):

    residuals_mc[:, j] = np.quantile(
        R[:, j],
        U[:, j]
    )

peaks_mc = (
    pred_day[None, :]
    + residuals_mc
).max(axis=1)

In [4]:
mc_mean = peaks_mc.mean()

mc_se = (
    peaks_mc.std(ddof=1)
    / np.sqrt(N_SIM)
)

mc_ci = (
    mc_mean - 1.96 * mc_se,
    mc_mean + 1.96 * mc_se
)

print(
    "Deterministic peak :",
    deterministic_peak
)

print(
    "Monte Carlo mean peak :",
    mc_mean
)

print(
    "Monte Carlo SE :",
    mc_se
)

print(
    "95% CI :", mc_ci
)

print(
    "Observed peak :",
    y_day.max()
)

Deterministic peak : 87882.06658735783
Monte Carlo mean peak : 87966.83785864351
Monte Carlo SE : 2.1648237169553477
95% CI : (np.float64(87962.59480415827), np.float64(87971.08091312875))
Observed peak : 87716.0


In [5]:
joblib.dump(
    {
        "deterministic_peak": deterministic_peak,
        "mc_mean": mc_mean,
        "mc_se": mc_se,
        "mc_ci": mc_ci,
        "n_sim": N_SIM
    },
    "models/block5_mc_baseline.joblib"
)

['models/block5_mc_baseline.joblib']

### Partie C — Analyse

L'approximation déterministe remplace l'incertitude par une valeur moyenne.

Cependant :

$$
E[\max(Y)]
\neq
\max(E[Y])
$$

Le Monte-Carlo conserve au contraire la variabilité des erreurs et permet d'estimer directement :

$$
E[\max(Y)]
$$

avec une erreur standard approximativement proportionnelle à :

$$
\frac{1}{\sqrt{N}}
$$

Cette convergence relativement lente motive l'utilisation de méthodes de réduction de variance et de Quasi-Monte Carlo.

### Conclusion de l'Étape 1

L'approximation déterministe donne :

$$
87882\text{ MW}
$$

alors que Monte-Carlo estime :

$$
E[Peak]\approx87967\text{ MW}
$$

L'approximation déterministe sous-estime donc ici le pic moyen simulé d'environ :

$$
85\text{ MW}
$$

L'erreur standard Monte-Carlo est faible :

$$
SE\approx2.16\text{ MW}
$$

La simulation probabiliste permet donc de prendre en compte l'effet de l'incertitude sur une quantité non linéaire comme le maximum.

## Étape 2 — Variance Reduction
### Réduction de variance par variables antithétiques

### Partie A — Mise en pratique

In [6]:
rng = np.random.default_rng(42)

N_PAIRS = 2500

U = rng.random(
    (N_PAIRS, 48)
)

resid_a = np.empty_like(U)
resid_b = np.empty_like(U)

for j in range(48):

    resid_a[:, j] = np.quantile(
        R[:, j],
        U[:, j]
    )

    resid_b[:, j] = np.quantile(
        R[:, j],
        1 - U[:, j]
    )

In [7]:
peaks_a = (
    pred_day[None, :]
    + resid_a
).max(axis=1)

peaks_b = (
    pred_day[None, :]
    + resid_b
).max(axis=1)

pair_means = (
    peaks_a + peaks_b
) / 2

In [8]:
ant_mean = pair_means.mean()

ant_se = (
    pair_means.std(ddof=1)
    / np.sqrt(N_PAIRS)
)

print(
    "Standard MC mean :",
    mc_mean
)

print(
    "Standard MC SE :",
    mc_se
)

print(
    "Antithetic mean :",
    ant_mean
)

print(
    "Antithetic SE :",
    ant_se
)

print(
    "SE reduction :",
    100 * (1 - ant_se / mc_se),
    "%"
)

Standard MC mean : 87966.83785864351
Standard MC SE : 2.1648237169553477
Antithetic mean : 87966.22733721773
Antithetic SE : 1.517128941062975
SE reduction : 29.919053954347095 %


In [9]:
joblib.dump(
    {
        "mean": ant_mean,
        "se": ant_se,
        "se_reduction_pct":
            100 * (1 - ant_se / mc_se)
    },
    "models/block5_antithetic.joblib"
)

['models/block5_antithetic.joblib']

### Partie C — Analyse

Les variables antithétiques cherchent à créer une corrélation négative entre deux simulations.

Pour une paire :

$$
U,\quad1-U
$$

on utilise l'estimateur :

$$
\frac{f(U)+f(1-U)}{2}
$$

Si les deux évaluations se compensent partiellement, la variance de l'estimateur diminue.

L'objectif est donc d'obtenir une erreur standard plus faible sans augmenter le nombre total de simulations.

### Conclusion de l'Étape 2

La méthode des variables antithétiques réduit l'erreur standard de :

$$
2.165
$$

à :

$$
1.517
$$

soit une réduction d'environ :

$$
29.9\%
$$

Cela correspond à une réduction de variance d'environ :

$$
50.9\%
$$

Les variables antithétiques sont donc efficaces pour cette estimation du pic moyen.

## Étape 3 — Quasi-Monte Carlo
### Séquences de Sobol

### Partie A — Mise en pratique

Le Monte Carlo classique utilise des points aléatoires.

Le Quasi-Monte Carlo cherche au contraire à couvrir l'espace :

$$
[0,1]^{48}
$$

de manière plus uniforme.

On utilise ici une séquence de Sobol randomisée.

In [10]:
from scipy.stats import qmc

N = 4096
M = 12
N_REP = 8

mc_estimates = []
sobol_estimates = []

for r in range(N_REP):

    # -------------------------
    # Monte Carlo classique
    # -------------------------

    rng = np.random.default_rng(100 + r)

    U_mc = rng.random((N, 48))

    resid_mc = np.empty_like(U_mc)

    for j in range(48):
        resid_mc[:, j] = np.quantile(
            R[:, j],
            U_mc[:, j]
        )

    peaks = (
        pred_day[None, :]
        + resid_mc
    ).max(axis=1)

    mc_estimates.append(
        peaks.mean()
    )

    # -------------------------
    # Sobol randomisé
    # -------------------------

    sampler = qmc.Sobol(
        d=48,
        scramble=True,
        seed=100 + r
    )

    U_sobol = sampler.random_base2(
        m=M
    )

    resid_sobol = np.empty_like(U_sobol)

    for j in range(48):
        resid_sobol[:, j] = np.quantile(
            R[:, j],
            U_sobol[:, j]
        )

    peaks = (
        pred_day[None, :]
        + resid_sobol
    ).max(axis=1)

    sobol_estimates.append(
        peaks.mean()
    )

In [11]:
mc_estimates = np.array(mc_estimates)
sobol_estimates = np.array(sobol_estimates)

comparison_qmc = pd.DataFrame({
    "Method": [
        "Monte Carlo",
        "Sobol QMC"
    ],
    "Mean estimate": [
        mc_estimates.mean(),
        sobol_estimates.mean()
    ],
    "Std across replications": [
        mc_estimates.std(ddof=1),
        sobol_estimates.std(ddof=1)
    ]
})

print(comparison_qmc)

        Method  Mean estimate  Std across replications
0  Monte Carlo   87966.920334                 2.450880
1    Sobol QMC   87965.917181                 0.016384


In [12]:
gain_qmc = 100 * (
    1
    -
    sobol_estimates.std(ddof=1)
    /
    mc_estimates.std(ddof=1)
)

print(
    "QMC variability reduction :",
    gain_qmc,
    "%"
)

QMC variability reduction : 99.33150906905522 %


In [13]:
joblib.dump(
    {
        "mc_estimates": mc_estimates,
        "sobol_estimates": sobol_estimates,
        "comparison": comparison_qmc
    },
    "models/block5_qmc_sobol.joblib"
)

['models/block5_qmc_sobol.joblib']

### Partie C — Analyse

Le Monte Carlo standard converge typiquement à un rythme :

$$
O(N^{-1/2})
$$

Le Quasi-Monte Carlo utilise des points mieux répartis dans l'espace des variables aléatoires.

Pour mesurer sa précision, on utilise plusieurs séquences Sobol randomisées et on compare la variabilité entre répétitions.

Si cette variabilité est plus faible que celle du Monte Carlo classique, Sobol fournit une estimation plus efficace pour un même budget de simulations.

### Conclusion de l'Étape 3

Avec le même budget de simulations, la séquence de Sobol produit une estimation beaucoup plus stable que le Monte Carlo classique.

La variabilité entre répétitions passe de :

$$
2.451
$$

à :

$$
0.016
$$

soit une réduction observée d'environ :

$$
99.3\%
$$

Les estimations moyennes restent très proches.

Le Quasi-Monte Carlo apparaît donc particulièrement efficace pour notre problème.

## Étape 4 — Numerical Convergence
### Comparaison de la convergence MC / Sobol

### Partie A — Mise en pratique

## Étape 4 — Numerical Convergence
### Comparaison de la convergence MC / Sobol

### Partie A — Mise en pratique

In [14]:
from scipy.stats import qmc

sizes = [256, 512, 1024, 2048, 4096]
N_REP = 8

results = []

for N in sizes:

    m = int(np.log2(N))

    mc_vals = []
    sobol_vals = []

    for r in range(N_REP):

        # Monte Carlo
        rng = np.random.default_rng(1000 + r)

        U_mc = rng.random((N, 48))

        resid_mc = np.empty_like(U_mc)

        for j in range(48):
            resid_mc[:, j] = np.quantile(
                R[:, j],
                U_mc[:, j]
            )

        mc_vals.append(
            (
                pred_day[None, :]
                + resid_mc
            ).max(axis=1).mean()
        )

        # Sobol
        sampler = qmc.Sobol(
            d=48,
            scramble=True,
            seed=1000 + r
        )

        U_sob = sampler.random_base2(m)

        resid_sob = np.empty_like(U_sob)

        for j in range(48):
            resid_sob[:, j] = np.quantile(
                R[:, j],
                U_sob[:, j]
            )

        sobol_vals.append(
            (
                pred_day[None, :]
                + resid_sob
            ).max(axis=1).mean()
        )

    results.append([
        N,
        np.std(mc_vals, ddof=1),
        np.std(sobol_vals, ddof=1)
    ])

convergence = pd.DataFrame(
    results,
    columns=[
        "N",
        "MC_std",
        "Sobol_std"
    ]
)

print(convergence)

      N    MC_std  Sobol_std
0   256  9.923834   0.408817
1   512  7.253798   0.247325
2  1024  6.082709   0.096060
3  2048  3.728426   0.040052
4  4096  3.053236   0.043014


In [15]:
convergence["gain_%"] = (
    100 *
    (
        1
        - convergence["Sobol_std"]
        / convergence["MC_std"]
    )
)

print(convergence)

      N    MC_std  Sobol_std     gain_%
0   256  9.923834   0.408817  95.880454
1   512  7.253798   0.247325  96.590405
2  1024  6.082709   0.096060  98.420774
3  2048  3.728426   0.040052  98.925768
4  4096  3.053236   0.043014  98.591214


In [16]:
joblib.dump(
    convergence,
    "models/block5_qmc_convergence.joblib"
)

['models/block5_qmc_convergence.joblib']

### Partie C — Analyse

Le Monte Carlo classique possède typiquement une erreur qui décroît comme :

$$
N^{-1/2}
$$

On vérifie empiriquement si Sobol réduit plus rapidement la variabilité de l'estimation lorsque le nombre de simulations augmente.

Une amélioration stable sur plusieurs valeurs de $N$ serait plus convaincante qu'un résultat obtenu pour une seule taille d'échantillon.

### Conclusion de l'Étape 4

La supériorité de Sobol ne dépend pas d'une seule taille d'échantillon.

Pour toutes les valeurs de $N$ testées, la variabilité est fortement inférieure à celle du Monte Carlo classique.

La réduction observée varie entre environ :

$$
95.9\%
$$

et :

$$
98.9\%
$$

Sobol permet donc d'obtenir une estimation beaucoup plus stable avec un nombre de simulations limité.

Le Quasi-Monte Carlo avec séquences de Sobol est retenu comme méthode de simulation numérique privilégiée pour ce problème.

## Étape 5 — Stochastic Approximation
### Estimation récursive d'un quantile par Robbins-Monro

### Partie A — Mise en pratique

La méthode de Robbins-Monro permet d'estimer une quantité sans conserver toutes les simulations en mémoire.

On l'utilise ici pour estimer récursivement le quantile 99 % du pic énergétique.

In [17]:
rng = np.random.default_rng(42)

N_SIM = 20000

U = rng.random(
    (N_SIM, 48)
)

resid = np.empty_like(U)

for j in range(48):
    resid[:, j] = np.quantile(
        R[:, j],
        U[:, j]
    )

peaks = (
    pred_day[None, :]
    + resid
).max(axis=1)

q99_reference = np.quantile(
    peaks,
    0.99
)

print(
    "Q99 reference :",
    q99_reference
)

Q99 reference : 88375.81369668136


In [18]:
p = 0.99

q_rm = np.median(peaks[:100])

trajectory = []

for n, x in enumerate(peaks, start=1):

    gamma = (
        200 / (n ** 0.60)
    )

    q_rm -= gamma * (
        (x <= q_rm) - p
    )

    if n % 500 == 0:
        trajectory.append([
            n,
            q_rm
        ])

print(
    "Q99 Robbins-Monro :",
    q_rm
)

print(
    "Q99 reference     :",
    q99_reference
)

print(
    "Absolute error    :",
    abs(q_rm - q99_reference)
)

Q99 Robbins-Monro : 88374.01472494937
Q99 reference     : 88375.81369668136
Absolute error    : 1.798971731986967


In [20]:
rm_results = pd.DataFrame(
    trajectory,
    columns=[
        "iteration",
        "q99_estimate"
    ]
)

joblib.dump(
    {
        "trajectory": rm_results,
        "q99_rm": q_rm,
        "q99_reference": q99_reference
    },
    "models/block5_robbins_monro.joblib"
)

['models/block5_robbins_monro.joblib']

### Partie C — Analyse

Robbins-Monro remplace un calcul global par une mise à jour récursive :

$$
q_{n+1}
=
q_n-\gamma_n
\left(
\mathbf 1_{X_n\le q_n}-p
\right)
$$

Il n'est donc pas nécessaire de conserver toutes les observations en mémoire.

Cette méthode devient particulièrement intéressante lorsque les simulations arrivent progressivement ou lorsque le volume de données est très important.

Le choix de la suite de pas $\gamma_n$ détermine le compromis entre vitesse d'adaptation et stabilité.

### Conclusion — Robbins-Monro

Robbins-Monro estime le quantile 99 % à :

$$
88374.0\text{ MW}
$$

contre une valeur de référence de :

$$
88375.8\text{ MW}
$$

L'erreur absolue est seulement :

$$
1.8\text{ MW}
$$

soit environ :

$$
0.002\%
$$

L'approximation stochastique permet donc ici d'estimer efficacement un quantile extrême par mises à jour successives.

## Étape 6 — Polyak-Ruppert Averaging
### Stabilisation de l'approximation stochastique

### Partie A — Mise en pratique

In [21]:
p = 0.99
burn = 5000

q = np.median(peaks[:100])

q_sum = 0.0
n_avg = 0

for n, x in enumerate(peaks, start=1):

    gamma = 200 / (n ** 0.60)

    q -= gamma * (
        (x <= q) - p
    )

    if n > burn:
        q_sum += q
        n_avg += 1

q_polyak = q_sum / n_avg

error_rm = abs(
    q - q99_reference
)

error_polyak = abs(
    q_polyak - q99_reference
)

print("Reference :", q99_reference)
print("Robbins-Monro :", q)
print("Polyak-Ruppert :", q_polyak)

print("RM error :", error_rm)
print("Polyak error :", error_polyak)

Reference : 88375.81369668136
Robbins-Monro : 88374.01472494937
Polyak-Ruppert : 88367.87144025866
RM error : 1.798971731986967
Polyak error : 7.942256422698847


In [22]:
joblib.dump(
    {
        "reference": q99_reference,
        "robbins_monro": q,
        "polyak_ruppert": q_polyak,
        "rm_error": error_rm,
        "polyak_error": error_polyak
    },
    "models/block5_polyak_ruppert.joblib"
)

['models/block5_polyak_ruppert.joblib']

### Partie C — Analyse

Polyak-Ruppert moyenne les itérations de Robbins-Monro après une période initiale :

$$
\bar q_N
=
\frac{1}{N-N_0}
\sum_{n=N_0+1}^{N}q_n
$$

L'objectif est de réduire la sensibilité aux fluctuations des dernières itérations.

Cette amélioration n'est pas nécessairement meilleure sur chaque réalisation, mais elle peut rendre l'estimation stochastique plus stable.

### Conclusion de l'Étape 6

Dans cette expérience, l'estimateur final de Robbins-Monro est plus précis que la moyenne de Polyak-Ruppert.

Les erreurs absolues sont :

$$
Erreur_{RM}\approx1.80\text{ MW}
$$

et :

$$
Erreur_{PR}\approx7.94\text{ MW}
$$

Polyak-Ruppert n'améliore donc pas l'estimation sur cette réalisation.

Cette méthode peut améliorer la stabilité asymptotique, mais son avantage dépend notamment du nombre d'itérations, du pas d'apprentissage et de la période de moyennage.

Pour notre problème, l'estimateur Robbins-Monro simple est conservé.

### Performance Summary — Bloc 5

| Méthode | Mesure principale | Résultat |
|---|---:|---:|
| Approximation déterministe | Peak estimate | 87882.07 MW |
| Monte Carlo | Mean peak | 87966.84 MW |
| Monte Carlo | Standard error | 2.165 MW |
| Antithetic | Standard error | 1.517 MW |
| Sobol QMC | Variabilité à N=4096 | 0.043 MW |
| Robbins-Monro | Q99 error | 1.80 MW |
| Polyak-Ruppert | Q99 error | 7.94 MW |

# Conclusion du Bloc 5

Ce bloc a étudié plusieurs méthodes d'approximation numérique et stochastique.

L'approximation déterministe sous-estime le pic moyen simulé d'environ :

$$
85\text{ MW}
$$

Le Monte Carlo permet d'intégrer explicitement l'incertitude.

Les variables antithétiques réduisent l'erreur standard d'environ :

$$
29.9\%
$$

et la variance d'environ :

$$
50.9\%
$$

Le Quasi-Monte Carlo avec séquences de Sobol fournit l'amélioration la plus importante, avec une réduction de variabilité supérieure à :

$$
95\%
$$

pour toutes les tailles d'échantillon étudiées.

Enfin, Robbins-Monro estime le quantile 99 % avec seulement :

$$
1.8\text{ MW}
$$

d'erreur par rapport à l'estimation de référence.

Sobol est donc retenu comme méthode privilégiée pour les simulations numériques, tandis que Robbins-Monro constitue une solution efficace pour l'estimation récursive de quantiles.